In [3]:
import pandas as pd
df=pd.read_csv('/content/IMDB.csv', encoding='latin1', engine='python', on_bad_lines='warn')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [6]:
import re
negation_words=[
    "not good","not bad","not great","don't like",
    "didn't like","never liked","wasn't good",
    "isn't good","no good"
]
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']"," ",text)

  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text


In [7]:
df['review']=df['review'].apply(clean_text)

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size=20000
max_len=250

tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)

X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [11]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
history=model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 412s 816ms/step - accuracy: 0.5522 - loss: 0.6669 - val_accuracy: 0.5651 - val_loss: 0.6552
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 441s 815ms/step - accuracy: 0.6017 - loss: 0.6091 - val_accuracy: 0.8211 - val_loss: 0.5385
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 467s 865ms/step - accuracy: 0.6996 - loss: 0.5514 - val_accuracy: 0.6856 - val_loss: 0.5806
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 463s 907ms/step - accuracy: 0.8200 - loss: 0.4313 - val_accuracy: 0.8367 - val_loss: 0.4226
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 472s 847ms/step - accuracy: 0.8736 - loss: 0.3252 - val_accuracy: 0.8602 - val_loss: 0.3503


In [13]:
loss, acc=model.evaluate(X_test_pad,y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 103ms/step - accuracy: 0.8612 - loss: 0.3502
Test Accuracy: 0.8611999750137329


In [14]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview:",review)
  print("Score:",prediction)

  if prediction>=0.5:
    print("Sentiment: Positive 😀")
  else:
    print('Sentiment: Negative 💩')

In [15]:
predict_sentiment("This movie was absolutely amazing and I loved")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 464ms/step

Review: this movie was absolutely amazing and i loved
Score: 0.9164749
Sentiment: Positive 😀


In [16]:
predict_sentiment("This movie was bad")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step

Review: this movie was bad
Score: 0.29033256
Sentiment: Negative 💩
